# LangChain Agent Under the Hood: Completing the ReAct Loop

Welcome back. In the previous lesson, we looked at how LangChain executes the first tool call and sends the result back to the LLM. Now, let's continue from that point and understand how the agent completes the reasoning loop.

Our example is:

**"What is 15 multiplied by 8, then divided by 3?"**

The agent has two tools:

* `multiply`
* `divide`

The important thing to understand is that the agent does not perform the entire calculation in one step. Instead, it reasons, calls a tool, observes the result, reasons again, calls another tool, and finally produces an answer.

This is the **ReAct pattern**:

> **Reason → Act → Observe → Reason → Act → Observe → Final Answer**

---

## Step 7: The Model Reasons Again

After LangChain executes the first tool call:

```text
multiply(15, 8)
```

the result is:

```text
120
```

LangChain sends this result back to the LLM as part of the updated conversation history.

The model now receives the complete context and reasons:

```text
15 × 8 = 120
```

But the original user question is not finished yet.

The user asked:

```text
15 × 8 ÷ 3
```

Therefore, the model realizes that it needs to perform another operation.

It decides:

```text
I need to divide 120 by 3.
```

So the model generates another structured tool call:

```json
{
  "name": "divide",
  "arguments": {
    "a": 120,
    "b": 3
  },
  "call_id": "call_002"
}
```

Notice something very important here.

The value `120` came from the result of the previous tool call.

The model did not receive `120` directly from the user. It received it from the previous step in the agent loop.

This is one of the most important characteristics of agentic systems:

> Each step can use the result of a previous step to determine what to do next.

This allows an agent to perform multi-step tasks dynamically.

---

# Step 8: LangChain Executes the Second Tool

Now the same process happens again.

LangChain receives the model's tool call request:

```text
Tool: divide
Arguments:
a = 120
b = 3
```

LangChain then performs several operations.

### 1. Parse the model response

LangChain identifies:

* Tool name: `divide`
* Argument `a`: `120`
* Argument `b`: `3`
* Tool call ID: `call_002`

### 2. Look up the tool

LangChain uses its internal tool registry to find the actual Python function associated with the name `divide`.

Conceptually:

```python
tool_registry["divide"]
```

returns the Python function:

```python
divide
```

### 3. Execute the Python function

LangChain calls:

```python
divide(120, 3)
```

The result is:

```text
40
```

So now the agent has completed both required operations:

```text
15 × 8 = 120
120 ÷ 3 = 40
```

The important point is that the second operation depends on the result of the first operation.

The flow is:

```text
User Question
     ↓
Multiply 15 × 8
     ↓
Result = 120
     ↓
Divide 120 ÷ 3
     ↓
Result = 40
```

---

# Step 9: The Updated Conversation History

After the second tool execution, LangChain now has an even larger conversation history.

Conceptually, the conversation contains:

### Message 1: User

```text
What is 15 multiplied by 8, then divided by 3?
```

### Message 2: Assistant

```text
Call multiply(15, 8)
```

### Message 3: Tool

```text
Result: 120
```

### Message 4: Assistant

```text
Call divide(120, 3)
```

### Message 5: Tool

```text
Result: 40
```

LangChain sends this complete conversation history back to the model.

This is important because LLMs are generally **stateless between individual API calls**.

The model does not automatically remember previous requests.

Therefore, LangChain provides the necessary context again so the model understands:

1. What the user originally asked.
2. Which tool it called first.
3. What the first tool returned.
4. Which tool it called second.
5. What the second tool returned.

The model now has everything it needs to answer the user.

---

# Step 10: The Model Produces the Final Answer

The model now examines the conversation.

It sees:

```text
15 × 8 = 120
120 ÷ 3 = 40
```

The original request has been completely satisfied.

Therefore, the model does not request another tool.

Instead, it returns a normal natural-language response.

For example:

```text
The answer is 40.
```

This time, the response contains content in the `content` field.

There are no more tool calls.

Conceptually:

```json
{
  "content": "The answer is 40.",
  "tool_calls": []
}
```

LangChain checks the response.

It asks:

```text
Does the response contain another tool call?
```

The answer is:

```text
No.
```

Then it asks:

```text
Does the response contain a final text response?
```

The answer is:

```text
Yes.
```

This tells LangChain that the agent has finished.

The ReAct loop stops.

---

# The Complete ReAct Loop

The complete execution now looks like this:

```text
User asks:
"What is 15 × 8 ÷ 3?"
        ↓
LLM reasons
        ↓
Calls multiply(15, 8)
        ↓
LangChain executes Python function
        ↓
Result = 120
        ↓
Result sent back to LLM
        ↓
LLM reasons again
        ↓
Calls divide(120, 3)
        ↓
LangChain executes Python function
        ↓
Result = 40
        ↓
Result sent back to LLM
        ↓
LLM reasons again
        ↓
No more tools required
        ↓
LLM generates final answer
        ↓
LangChain returns result
        ↓
Your Python code prints the answer
```

This is the complete **Reason → Act → Observe** cycle.

The agent performs:

```text
Reason
  ↓
Act
  ↓
Observe
  ↓
Reason
  ↓
Act
  ↓
Observe
  ↓
Final Answer
```

In this example, the loop executes twice because two tool operations are required.

However, the same architecture can support much more complicated tasks.

For example:

```text
User asks a question
        ↓
Search the web
        ↓
Read the results
        ↓
Query a database
        ↓
Analyze the data
        ↓
Call another API
        ↓
Compare the results
        ↓
Generate final answer
```

The underlying pattern remains the same.

---

# LangChain Returns the Final Result

Once the model produces the final answer, LangChain returns the assembled result to your Python code.

Your code can then print the result:

```python
print(result)
```

The user might only see:

```text
The answer is 40.
```

However, behind that simple answer, many things happened:

* The LLM analyzed the question.
* LangChain provided the available tools.
* The LLM requested the `multiply` tool.
* LangChain parsed the tool call.
* LangChain found the Python function.
* The Python function executed.
* The result `120` was returned.
* The result was added to the conversation.
* The LLM analyzed the result.
* The LLM requested the `divide` tool.
* LangChain executed the `divide` function.
* The result `40` was returned.
* The LLM analyzed the complete conversation.
* The LLM generated the final answer.
* LangChain returned the result to your application.

All of this can happen behind a single line:

```python
agent.invoke(...)
```

---

# What LangChain Does Behind the Scenes

When you write something like:

```python
agent = create_agent(
    model=model,
    tools=tools
)

result = agent.invoke(...)
```

it may look extremely simple.

However, LangChain is handling a significant amount of complexity for you.

It is responsible for:

### 1. Creating Tool Schemas

LangChain examines your Python functions, decorators, type hints, and docstrings and converts them into structured tool definitions.

### 2. Formatting Messages

LangChain formats messages according to the requirements of the model API.

### 3. Calling the Model API

It sends requests to the LLM provider.

### 4. Parsing Responses

It examines the model's response to determine whether it contains:

* A tool call
* A final answer
* Both or neither

### 5. Looking Up Tools

It maps the tool name provided by the LLM to the actual Python function.

For example:

```text
"multiply"
      ↓
Python multiply() function
```

### 6. Executing Tools

It calls the appropriate Python function with the arguments generated by the model.

### 7. Returning Results

It adds the tool result to the conversation history.

### 8. Repeating the Loop

It sends the updated conversation back to the model and continues the process when more work is required.

### 9. Detecting Completion

When there are no more tool calls and the model provides a final response, LangChain stops the agent loop.

### 10. Returning the Final Result

Finally, LangChain returns the completed response to your application.

---

# Why Understanding the Internals Matters

Frameworks like LangChain make agent development much easier.

Instead of manually implementing all of this logic, you can write only a few lines of code.

However, as your applications become more complex, understanding what happens underneath becomes increasingly important.

This knowledge helps when:

* Debugging failed tool calls
* Understanding unexpected agent behavior
* Creating custom agent loops
* Improving performance
* Reducing unnecessary model calls
* Managing conversation history
* Controlling tool access
* Building production-ready systems
* Monitoring agent execution

For example, if an agent repeatedly calls the wrong tool, you need to understand that the model is selecting tools based on the tool descriptions and schemas.

If an agent loses context, you need to understand how conversation history is being managed.

If a tool fails, you need to understand the handoff between the LLM, LangChain, and your Python code.

Therefore, using a framework is valuable, but understanding the framework's internals is equally important.

---

# The Complete Architecture

The complete flow can be visualized as three layers:

```text
┌─────────────────────────┐
│      Your Python Code   │
│                         │
│  agent.invoke(question) │
└────────────┬────────────┘
             │
             ↓
┌─────────────────────────┐
│        LangChain        │
│                         │
│  Parse tool calls       │
│  Find tools             │
│  Execute Python code    │
│  Manage conversation    │
│  Control agent loop     │
└────────────┬────────────┘
             │
             ↓
┌─────────────────────────┐
│          LLM            │
│                         │
│  Understand question    │
│  Reason about next step │
│  Select tools            │
│  Generate final answer  │
└─────────────────────────┘
```

The LLM does the reasoning.

LangChain manages the orchestration.

Your Python application controls the overall system.

The tools perform real-world actions.

---

# Final Takeaway

The most important concept from this lesson is that an agent is not simply an LLM responding to a user.

It is a continuous interaction between:

```text
Your Application
       ↕
LangChain
       ↕
LLM
       ↕
Tools
```

The LLM decides what should happen next.

LangChain interprets that decision and executes the appropriate action.

The tool performs the actual operation.

The result is sent back to the LLM.

The LLM reasons again.

This process repeats until the task is complete.

In our example:

```text
15 × 8 ÷ 3
```

the agent dynamically performed:

```text
Reason → Multiply → Observe 120
                 ↓
          Reason → Divide → Observe 40
                              ↓
                       Final Answer
```

Nobody explicitly programmed the agent to always multiply first and divide second.

The LLM understood the user's request, selected the appropriate tools, used the result of one operation as the input to the next, and decided when the task was complete.

That dynamic decision-making is what makes an agent different from a fixed workflow.

The example may be simple, but the architecture is exactly the same for more advanced applications involving APIs, databases, web search, file systems, enterprise applications, and other real-world tools.

Understanding this flow is therefore essential for anyone who wants to debug, customize, and build production-grade AI agent systems.
